In [ ]:
# Add the parent directory of the current working directory to the Python path at runtime. 
# In order to import modules from the src directory.
import os
import sys 


current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)



In [102]:
import json
import tqdm
import openai
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers.json import SimpleJsonOutputParser

from src.utils.semantic import load_configurations, safe_dictionary_extraction, get_abstract_strings

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
openai.api_key = os.environ["OPENAI_API_KEY"]

BASEPATH = os.environ['BASEPATH']

In [ ]:
def merge_sections(abstract):
    relevant_sections = ['Introduction', 'Methods', 'Results', 'Conclusion']
    merged_sections = ''
    for section in abstract['sections']:
        if section['label'] in relevant_sections:
            merged_sections = '\n'.join([merged_sections, section['markdown']])
    return merged_sections


In [ ]:
submissions_file = '/media/mario/SSD/Projects/ohbm2026/ui/data/abstracts.detail.json'

with open(submissions_file, 'r') as f:
    submissions = json.load(f)['abstracts']
submission_ids = list(submissions.keys())

abstracts = [merge_sections(submissions[submission_id]) for submission_id in submission_ids]

In [130]:
llm = ChatOpenAI(temperature=1.0, model_name='gpt-5.2')

DIMENSIONS_PROMPT = PromptTemplate(
    input_variables=["abstract"],
    template="""
            You are an expert in neuroscience and scientific text analysis. 
            You are provided a neuroscientific abstract.
            
            Your task is to identify **key neuroscience dimensions** that best describe the abstract, guided by the following 9 dimensions:

            1. Appliedness: The extent to which the research is basic science (fundamental mechanisms) or applied (translational, clinical, legal, neuroeconomics, method development, advancing technology).
            2. Modality: The sensory and/or motor modality under investigation (e.g., visual, auditory, gustatory, somatosensory, motor, sensorimotor, multimodal).
            3. Spatiotemporal Scale: The spatial (e.g., molecular, cellular, circuit, region, systems, whole-brain) and temporal (e.g., microsecond, millisecond, second, minute, hour, day, week, month, year, lifetime) scale of the research.
            4. Cognitive Complexity: The level of cognitive complexity under investigation from low level (e.g., sensory processing, motor control) to high level (e.g., language, decision making, social cognition).
            5. Species: The species under investigation (e.g., human, non-human primate, rodent, drosophila, zebrafish, C. elegans).
            6. Theory Engagement: The extent to which the research is theory-driven (hypothesis testing) or data-driven (exploratory, descriptive).
            7. Theory Scope: The scope of the theory under investigation, ranging from specific mechanisms to broad overarching theories of the brain. Intermediate between these are theories focusing on pathophysiology of a specific disorder and highly influential theories with narrow domain coverage. Indiciate the specific unifying theoretical frameworks (e.g., Predictive Coding, Critical Brain Hypothesis, Communication through Coherence, Free Energy Principle, Active Inference, Global Neuronal Workspace, Integrated Information Theory, etc.) if applicable (else, no general theory). There might be more than one framework.
            8. Methodological Approach: The methodological approach used in the research (e.g., experimental, computational, theoretical, modeling, simulation, data analysis, review, meta-analysis, etc.). Identify specific methods if applicable (e.g., optogenetics, fMRI, EEG, MEG, TMS, lesion studies, single-unit recordings, etc.). There might be more than one method.
            9. Interdisciplinarity: The extent to which the research is interdisciplinary, combining methods and concepts from multiple fields (e.g., medicine, biology, chemsitry, psychology, computer science, physics, engineering, mathematics, philosophy).

            Examples are not exhaustive and an abstract may contain multiple dimensions.

            **Output Format:**

            Please present your findings in **JSON format** with the following structure:
            {{
                "Dimension 1 - Appliedness": "Brief assessment of the research's appliedness.",
                "Dimension 2 - Modality": "Brief overview of the sensory and/or motor modality under investigation.",
                "Dimension 3 - Spatiotemporal Scale": "Brief description of the spatial and temporal scale of the research.",
                "Dimension 4 - Cognitive Complexity": "Brief assessment of the research's cognitive complexity.",
                "Dimension 5 - Species": "Brief overview of the species under investigation.",
                "Dimension 6 - Theory Engagement": "Brief assessment of the research's theory engagement.",
                "Dimension 7 - Theory Scope": "Brief assessment of the research's theory scope. Identify specific theoretical frameworks if applicable. There might be more than one framework. Not all articles need to fall under any specific framework, but a significant portion should (one or two are not enough).",
                "Dimension 8 - Methodological Approach": "Brief overview of the methodological approach used in the research. Identify specific methods if applicable. There might be more than one method.",
                "Dimension 9 - Interdisciplinarity": "Brief assessment of the research's interdisciplinarity."
            }}

            ```

            **Instructions:**
            - **Accuracy is crucial**: Ensure all information is directly supported by the provided abstract. Do not include information not present in the abstract or make external assumptions.

            - **Clarity and Precision**: Assessments and descriptions should be clear and accurately reflect the content of the abstract.

            - **Conciseness**: Do not include any additional text or explanations beyond the specified JSON output. Do not generate more output than necessary.

            - **Compliance**: Return the JSON file even when you did not receive any abstracts. Just say not applicable for all dimensions.

            **Here is the abstract:**
            {abstract}""",
)

CATEGORIES_PROMPT = PromptTemplate(
    input_variables=["dimension", "categories", "analysis"],
    template="""
            You are an expert in neuroscience. 
            You are provided with an analysis of research within a neuroscientific abstract along the following 9 dimensions:

            1. Appliedness: The extent to which the research is basic science (fundamental) or applied in one of several ways.
            2. Modality: The sensory and/or motor modality under investigation.
            3. Spatiotemporal Scale: The spatial and temporal scale of the research. Can vary from microscale to macroscale for both space and time.
            4. Cognitive Complexity: The level of cognitive complexity under investigation from low level (e.g., sensory processing, motor control) to high level (e.g., language, decision making, social cognition).
            5. Species: The species under investigation.
            6. Theory Engagement: The extent to which the research is theory-driven (hypothesis testing) or data-driven (exploratory, descriptive).
            7. Theory Scope: The scope of the theory under investigation. Overarching Framework: In neuroscience, an overarching framework is a broad theoretical approach that aims to explain fundamental principles of brain function across multiple cognitive and neural domains.
                            Domain Framework: In neuroscience, a domain framework is an integrative theory focusing on a specific subfield, offering cohesive principles for that area.
                            Disease-specific Framework: In neuroscience, a disease-specific framework is a theory that details the neural causes, mechanisms, and manifestations of a particular neurological or psychiatric condition.
                            Micro Theory: In neuroscience, a micro theory is a narrowly scoped, mechanistic account or model that explains one specific process or phenomenon in the brain.
            8. Methodological Approach: The methodological approach used in the research. Experimental: Studies in which researchers deliberately manipulate one or more variables under controlled conditions to test for causal effects.
                                        Observational (Correlational / Descriptive): Studies that measure variables in naturally occurring settings without introducing any active intervention, focusing on describing or correlating observed phenomena.
                                        Computational / coding: Studies that construct or test mathematical, algorithmic, or simulation-based models to predict, explain, or interpret empirical data or biological processes.
                                        Theoretical / Conceptual: Work that develops, refines, or critiques conceptual frameworks and theories without generating new empirical data or running computational simulations.
                                        Meta-Analytic / Systematic Review: Research that synthesizes and reanalyzes existing primary studies, systematically aggregating findings using quantitative (meta-analysis) or rigorous protocol-based (systematic review) methods.
            9. Interdisciplinarity: The extent to which the research is interdisciplinary, combining methods and concepts from multiple fields. From low (confined to a single discipline) to very high (incorporating multiple disciplines in a transcdisciplinary manner).
                                    Multidisciplinary: Multiple disciplines study the same problem in parallel, each applying its own methods and perspectives but with little cross-integration.
                                    Interdisciplinary: Researchers from different disciplines integrate theories, methods, or data to create shared frameworks or solutions that transcend any single field.
                                    Transdisciplinary: Collaboration goes beyond standard academic boundaries, involving non-academic stakeholders or merging disciplines so completely that new fields or holistic approaches emerge.

            Your task is to focus solely on the dimension of {dimension} and provide a binary indication ("yes" or "no") of whether the research within the abstract falls within the specified categories.
            Here are the categories for this dimension:
            {categories}

            **Output Format:**

            Please present your findings in **JSON format** with the following structure:
            {{
                "Category 1": "yes" / "no",
                "Category 2": "yes" / "no",
                "Category 3": "yes" / "no",
                ...
                category n: "yes" / "no"
            }}

            ```

            **Instructions:**
            - **Accuracy is crucial**: Ensure all information is directly supported by the provided analysis. Do not include information not present in the analysis or make external assumptions.

            - **Consisteny**: Ensure that the evaluation of the dimension does not contradict the provided analysis (including other dimensions).
            
            - **Focus and Precision**: Only evaluate the dimension of {dimension} and provide a binary response for each category. Do not include any additional information or explanations.

            - **Proper Category Naming**: Ensure that the categories are named correctly and accurately reflect the content of the analysis. ONLY use the provided categories and replace Category 1, Category 2, etc. with the actual category names.

            - **Binary Response**: Ensure that the response for each category is binary (yes or no) and does not include any other text or explanations.


            **Here is the analysis of the abstract along the dimension of {dimension}:**
            {analysis}""",
)

dimensions_chain = DIMENSIONS_PROMPT | llm | SimpleJsonOutputParser()
categories_chain = CATEGORIES_PROMPT | llm | SimpleJsonOutputParser()

In [117]:
# Dimensions (qualitative)

required_fields = [
                'Dimension 1 - Appliedness', 'Dimension 2 - Modality',
                'Dimension 3 - Spatiotemporal Scale',
                'Dimension 4 - Cognitive Complexity', 'Dimension 5 - Species',
                'Dimension 6 - Theory Engagement',
                'Dimension 7 - Theory Scope',
                'Dimension 8 - Methodological Approach',
                'Dimension 9 - Interdisciplinarity'
        ]

# check if /media/mario/HDD/Data/NeuroScape/OHBM2026/abstract_dimensions.csv exists, if so, load it and skip the extraction process

if os.path.exists('/media/mario/HDD/Data/NeuroScape/OHBM2026/abstract_dimensions.csv'):
    abstract_dimensions_df = pd.read_csv('/media/mario/HDD/Data/NeuroScape/OHBM2026/abstract_dimensions.csv')
    abstract_dimensions = abstract_dimensions_df.to_dict(orient='records')
    print(f"Loaded {len(abstract_dimensions)} abstract dimensions from CSV file.")
    processed_ids = np.array([abstract_dimension['id'] for abstract_dimension in abstract_dimensions])
else:
        abstract_dimensions = []

for abstract in tqdm.tqdm(abstracts, total=len(abstracts)):

        submission_id = submission_ids[abstracts.index(abstract)]
        if int(submission_id) in processed_ids:
                continue


        chain_input = {"abstract": abstract}
        extracted_dict = safe_dictionary_extraction(
                required_fields, chain_input, dimensions_chain,
                3, 0.2)


        abstract_dimensions_dict = {
                'id': submission_id,
                'Dimensions':
                '\n'.join(
                        [f'{key}: {value}' for key, value in extracted_dict.items()])
                }

        abstract_dimensions.append(abstract_dimensions_dict)


Loaded 3268 abstract dimensions from CSV file.


100%|██████████| 3333/3333 [3:40:42<00:00,  3.97s/it]  


In [ ]:
abstract_dimensions_df = pd.DataFrame(abstract_dimensions)
abstract_dimensions_df.to_csv('/media/mario/HDD/Data/NeuroScape/OHBM2026/abstract_dimensions.csv', index=False)

In [121]:
dimension_categories = {
        'Appliedness': [
            'Fundamental', 'Translational', 'Clinical', 'Legal', 'Economic',
            'Method Development', 'Technological Exploitation'
        ],
        'Modality': [
            'Auditory', 'Visual', 'Olfactory', 'Gustatory', 'Somatosensory',
            'Multimodal', 'Visuomotor', 'Sensorimotor', 'Motor', 'Emotional',
            'Behavioral', 'Cognitive'
        ],
        'Spatial Scale': [
            'Molecular', 'Cellular', 'Circuit', 'Region', 'Systems',
            'Whole-brain'
        ],
        'Temporal Scale': [
            'Microsecond', 'Millisecond', 'Second', 'Minute', 'Hour', 'Day',
            'Week', 'Month', 'Year', 'Lifetime'
        ],
        'Cognitive Complexity':
        ['Low-level Sensory', 'Low-level Motor', 'Mid-level', 'High-level'],
        'Species': [
            'Human', 'Non-human primates', 'Rodents', 'Mammals', 'Birds',
            'Fish', 'Amphibians', 'Invertebrates', 'Cell cultures', 'Other'
        ],
        'Theory Engagement': ['Data-driven', 'Hypothesis-driven'],
        'Theory Scope': [
            'Overarching Framework', 'Domain Framework',
            'Disease-specific Framework', 'Micro Theory'
        ],
        'Methodological Approach': [
            'Experimental', 'Observational', 'Computational', 'Theoretical',
            'Meta-analytic'
        ],
        'Interdisciplinarity': ['Low', 'Medium', 'High', 'Very High']
    }

In [128]:
analysis = abstract_dimensions_df.iloc[1]['Dimensions']

analysis_dict = {item.split(':')[0].strip(): item.split(':')[1].strip() for item in analysis.split('\n') if ': ' in item}

analysis_dict['Dimension 1 - Appliedness']

'Translational/methods-focused clinical neuroscience'

In [131]:
# Appliedness (quantitative)

analysis = abstract_dimensions_df.iloc[1]['Dimensions']
analysis_dict = {item.split(':')[0].strip(): item.split(':')[1].strip() for item in analysis.split('\n') if ': ' in item}


chain_input = {"dimension": 'Appliedness', "categories": dimension_categories['Appliedness'], "analysis": analysis_dict['Dimension 1 - Appliedness']}
extracted_dict = safe_dictionary_extraction(dimension_categories['Appliedness'], chain_input, categories_chain,3, 0.2)

In [132]:
extracted_dict

{'Fundamental': 'no',
 'Translational': 'yes',
 'Clinical': 'yes',
 'Legal': 'no',
 'Economic': 'no',
 'Method Development': 'yes',
 'Technological Exploitation': 'no'}

In [3]:

import os  
import sys



import openai

import numpy as np
import pandas as pd

from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers.json import SimpleJsonOutputParser

from src.utils.semantic import load_configurations, safe_dictionary_extraction, get_abstract_strings

from dotenv import load_dotenv, find_dotenv

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)
load_dotenv(find_dotenv())
openai.api_key = os.environ["OPENAI_API_KEY"]
voyage_api_key = os.environ["VOYAGE_API_KEY"]
BASEPATH = os.path.join(os.path.expanduser("~"), "Documents", "OHBM2026")

In [ ]:
# markdown files
phenomena_file = os.path.join(BASEPATH, "ohbm_phenomena_deep_dive.md")
with open(phenomena_file, "r") as f:
    phenomena_text = f.read()

theories_file = os.path.join(BASEPATH, "ohbm_theories.md")
with open(theories_file, "r") as f:
    theories_text = f.read()

definitions_file = os.path.join(BASEPATH, "definitions.md")   
with open(definitions_file, "r") as f:
    definitions_text = f.read()

# csf file
abstract_file = os.path.join(BASEPATH, "abstracts.csv")
abstract_df = pd.read_csv(abstract_file)

id                                                         1176971
abstract_body    test\n\ntest\n\ntest\n\ntest\n\ntest\n\ntest\n...
phenomena                                                      NaN
theories                                                       NaN
Name: 0, dtype: object

In [20]:
abstract_df.head()

,id,abstract_body,phenomena,theories
0,1176971,test\n\ntest\n\ntest\n\ntest\n\ntest\n\ntest\n...,NaN,NaN
1,1196698,Transcranial magnetic stimulation (TMS) has pr...,PHEN_017,NaN
2,1196735,The dorsal striatum (comprising the caudate nu...,PHEN_017|PHEN_021|PHEN_030|PHEN_049,NaN
3,1196743,Effective connectivity (EC) studies have revea...,PHEN_004|PHEN_006|PHEN_007|PHEN_017|PHEN_020|P...,NaN
4,1196749,Dedifferentiation is an age-related process ma...,PHEN_017|PHEN_025,NaN


In [33]:
llm = ChatOpenAI(temperature=1.0,model_name='gpt-5.2')


In [130]:
PHENOM_EXTRACTION_PROMPT = PromptTemplate(
    input_variables=["abstract","definitions","phenomena"],  
    template="""
You are an expert neuroscientist and philosopher of science. Your task is to identify which stable "Scientific Phenomena" are investigated in the provided abstract.

### Philosophical Framework:
- **Phenomena** are stable, recurring features of the brain/behavior (e.g., "Small-World Network Topology").
- **Data** are the messy, specific records of an experiment (e.g., "a 3T Siemens scanner," "p < 0.05").
- **Your Goal**: Identify the "Gold" (Phenomena) being studied, not the "Dirt" (Data/specific methods).

### Available Phenomena (Titles & IDs):
{phenomena}

### Detailed Definitions:
{definitions}

 ### Guidelines:
   18 1. **Focus on Investigation**: Only select a phenomenon if the abstract seeks to explain, measure, or model it.
   19 2. **Be Exhaustive but Precise**: Identify ALL matching phenomena from the list.
   20 3. **Format**: Your final output must be EXCLUSIVELY a JSON object. No preamble or pipe-separated lists.
   21
   22 ### Abstract to Analyze:
   23 {abstract}
   24
   25 ### Instructions:
   26 1. **Analyze**: Identify the core biological or psychological features being tested.
   27 2. **Reasoning**: For each matching phenomenon, provide a short justification.
   28 3. **Output**: Return a JSON object with a "matches" key containing a list of phenomena:
   29 {{
   30    "matches": [
   31       {{
   32          "id": "PHEN_XX",
   33          "title": "Phenomenon Title",
   34          "justification": "Short explanation of how the abstract investigates this feature."
   35       }}
   36    ]
   37 }}
   38
   39 If no phenomena match, return: {{ "matches": [] }}
   40 Only return the JSON object.

""",

)

THEORY_EXTRACTION_PROMPT = PromptTemplate(
    input_variables=["abstract","definitions", "theories"],
    template="""
You are an expert neuroscientist and philosopher of science. Your task is to identify which theoretical frameworks are invoked or tested in the provided abstract.

### Philosophical Framework:
- **Theories** are systematic explanations of phenomena (e.g., "The Neural Mass Model").
- **Data** are the messy, specific records of an experiment (e.g., "a 3T Siemens scanner," "p < 0.05").
- **Your Goal**: Identify the "Gold" (Theories) being studied, not the "Dirt" (Data/specific methods).

### Available Theories (Titles & IDs):
{theories}

### Detailed Definitions:
{definitions}

 ### Guidelines:
   18 1. **Focus on Investigation**: Only select a theory if the abstract describes research that clearly tests a theory of follows its explantory schemata.
   19 2. **Be Exhaustive but Precise**: Identify ALL matching theories from the list.
   20 3. **Format**: Your final output must be EXCLUSIVELY a JSON object. No preamble or pipe-separated lists.
   21
   22 ### Abstract to Analyze:
   23 {abstract}
   24
   25 ### Instructions:
   26 1. **Analyze**: Identify whether the abstract describes research that clearly tests a theory of follows its explantory schemata. Identify which one(s).
   27 2. **Reasoning**: For each matching theory, provide a short justification.
   28 3. **Output**: Return a JSON object with a "matches" key containing a list of theories:
   29 {{
   30    "matches": [
   31       {{
   32          "id": "THEO_XX",
   33          "title": "Theory Title",
   34          "justification": "Short explanation of why you judged that the abstract investigates this theory."
   35       }}
   36    ]
   37 }}
   38
   39 If no theories match, return: {{ "matches": [] }}
   40 Only return the JSON object.

""",
)

THEORY_EVAL_PROMPT = PromptTemplate(
    input_variables=["abstract","definitions", "theories"],
    template="""
    You are an expert neuroscientist and philosopher of science. Your task is to rigorously evaluate whether a given scientific abstract is genuinely **theory-driven**. 

Many papers superficially name-drop theoretical frameworks in their introduction or conclusion. Your goal is to separate these casual mentions from research that is structurally driven by a theory's explanatory schema.

### Philosophical Framework:
- **Theory-Driven Research**: The theory provides the "explanatory engine" for the study. The study explicitly tests the theory's predictions, or its hypotheses, experimental design, and data interpretation are unambiguously structured around the theory's specific logic and vocabulary (e.g., "prediction errors" in Predictive Coding).
- **Casual Mention (Reject)**: The theory is merely referenced as background context, a post-hoc speculation in the discussion, or a generic buzzword without influencing the actual scientific investigation.

### Candidate Theories to Evaluate:
{theories}

### Detailed Definitions & Explanatory Schemas:
{definitions}

### Guidelines for Rigorous Evaluation:
1. **Demand Structural Evidence**: To accept a match, you must find evidence that the theory's specific explanatory schema drives the research question, the operationalization of variables, or the core mechanistic explanation of the results.
2. **Reject Post-Hoc Speculation**: If the theory is only mentioned as a possible explanation at the end (e.g., "These results might be consistent with the Global Workspace Theory"), reject it.
3. **Format**: Your output must be EXCLUSIVELY a JSON object. Do not include markdown formatting like ```json or any preamble.

### Abstract to Analyze:
{abstract}

### Instructions:
1. **Analyze**: Evaluate the abstract against the provided candidate theories. Does the study explicitly test the theory or rigorously follow its explanatory schema?
2. **Reasoning Step**: For each candidate theory, explicitly state *how* the study's design or hypothesis relies on the theory's schema. If it's just a name-drop, state that.
3. **Final Decision**: Assign a boolean `is_theory_driven` (true/false).
4. **Output**: Return a JSON object with a "matches" key.

{{
   "matches": [
      {{
         "id": "THEO_XX",
         "title": "Theory Title",
         "evaluation": "Explain specifically how the theory's explanatory schema structures the study's hypothesis/methods, or explain why it is merely a superficial mention.",
         "is_theory_driven": True/False
      }}
   ]
}}

If no candidate theories are genuinely driving the research, return the theories with `is_theory_driven: false`, or if none even loosely apply, return: {{ "matches": [] }}

    """,
)

In [131]:
phenomena_chain = PHENOM_EXTRACTION_PROMPT | llm | SimpleJsonOutputParser()
theory_chain = THEORY_EXTRACTION_PROMPT | llm | SimpleJsonOutputParser()
eval_chain = THEORY_EVAL_PROMPT | llm | SimpleJsonOutputParser()

In [102]:
required_fields = ["matches"]

retries = 5
delay = 0.2



In [125]:
def extract_phenomena(dict):
    phenomena = [match["id"] for match in dict["matches"]]
    phenomena = '|'.join(phenomena)
    return phenomena

def extract_theories(dict):
    theories = [match["id"] for match in dict["matches"]]
    theories = '|'.join(theories)
    return theories

def extract_theory_evals(dict):
    # return true if at least one theory is judged to be genuinely driving the research, otherwise false
    for match in dict["matches"]:
        if match["is_theory_driven"]:
            return True
    return False


In [97]:
phenomena = []

for index, row in abstract_df.iterrows():
    abstract_text = row["abstract_body"]
    
    chain_input = {
        "abstract": abstract_text,
        "definitions": definitions_text,
        "phenomena": phenomena_text
    }
    
    result = safe_dictionary_extraction(required_fields, chain_input, phenomena_chain, retries, delay)
    extracted_phenomena = extract_phenomena(result)
    phenomena.append(extracted_phenomena)


new_abstract_df = abstract_df.copy()

new_abstract_df["phenomena"] = phenomena
new_abstract_df.to_csv(os.path.join(BASEPATH, "abstracts_with_phenomena.csv"), index=False)


In [ ]:
theories = []

for index, row in abstract_df.iterrows():
    abstract_text = row["abstract_body"]
    
    chain_input = {
        "abstract": abstract_text,
        "definitions": definitions_text,
        "theories": theories_text
    }
    
    result = safe_dictionary_extraction(required_fields, chain_input, theory_chain, retries, delay)
    extracted_theories = extract_theories(result)
    theories.append(extracted_theories)

new_abstract_df["theories"] = theories
new_abstract_df.to_csv(os.path.join(BASEPATH, "abstracts_with_phenomena_with_theories.csv"), index=False)


In [123]:
from tqdm import tqdm

In [139]:
for index, row in tqdm(new_abstract_df.iterrows(), total=new_abstract_df.shape[0]):
    abstract_text = row["abstract_body"]
    theories = row["theories"]

    has_theories = bool(theories.strip())
    if not has_theories:
        continue
    
    chain_input = {
        "abstract": abstract_text,
        "definitions": definitions_text,
        "theories": theories_text
    }
    
    result = safe_dictionary_extraction(required_fields, chain_input, eval_chain, retries, delay)

    is_theory_driven = extract_theory_evals(result)

    if is_theory_driven:
        extracted_theories = extract_theories(result)
        new_abstract_df.at[index, "theories"] = extracted_theories
    else:
        new_abstract_df.at[index, "theories"] = ""

new_abstract_df.to_csv(os.path.join(BASEPATH, "abstracts_with_phenomena_with_theories_evaluated.csv"), index=False)

  0%|          | 0/3333 [00:00<?, ?it/s]

  5%|▌         | 183/3333 [18:39:05<321:03:05, 366.92s/it]


KeyboardInterrupt: 

In [140]:
result

{'matches': [{'id': 'THEO_009',
   'title': 'Excitation-Inhibition (E/I) Imbalance & Neural Noise Theory',
   'evaluation': 'The abstract is compatible with E/I imbalance accounts of schizophrenia (e.g., increased high-frequency events and links to NMDA receptor availability), but it does not operationalize E/I balance, neural noise, or excitation/inhibition ratio, nor does it derive specific, discriminating predictions from that schema. The hypotheses (increased ripple occurrence, altered cortical distribution, reduced hippocampo–cortical transitions) are framed in terms of ripple phenomenology and coupling, not in terms of E/I gain, inhibitory dysfunction, signal-to-noise, or mechanistic tests (e.g., GABA/Glx measures, computational noise parameters, pharmacological manipulation). The PET–MEG NMDA mention is background context from prior work, not a theoretical engine structuring this study’s design/analysis.',
   'is_theory_driven': False},
  {'id': 'THEO_008',
   'title': 'Triple N

In [121]:
abstract_text = new_abstract_df.iloc[4]['abstract_body']

chain_input = {
    "abstract": abstract_text,
    "definitions": definitions_text,
    "theories": theories_text
}

result = safe_dictionary_extraction(required_fields, chain_input, eval_chain, retries, delay)
result

{'matches': [{'id': 'THEO_005',
   'title': 'Entropic Brain Hypothesis',
   'evaluation': 'The abstract uses the term “entropy,” but operationalizes it as a graph-theoretic Shannon entropy over edge-community participation derived from resting-state functional connectivity, explicitly as an index of dedifferentiation/specialization. THEO_005 (Entropic Brain Hypothesis) is a theory linking conscious experience and global brain dynamical entropy/criticality (e.g., psychedelics vs anesthesia) to changes in the richness of consciousness. This study does not investigate altered states of consciousness, does not frame entropy as a mechanism for conscious experience, and does not test any entropic-brain predictions (e.g., shifts toward criticality, changes in experiential richness, pharmacological manipulation). Therefore, the overlap is terminological/metric-level rather than theory-level: the theory’s explanatory engine is not used to generate hypotheses, guide design, or interpret results.

In [84]:
abstract_text = abstract_df.loc[0, "abstract_body"]

chain_input = {
    "abstract": abstract_text,
    "definitions": definitions_text,
    "phenomena": phenomena_text
}

result = safe_dictionary_extraction(required_fields, chain_input, phenomena_chain, retries, delay)

In [98]:
result

{'matches': [{'id': 'PHEN_023',
   'title': 'Neurodevelopmental Growth Trajectories',
   'justification': 'The study tracks juvenile-to-adult maturation (ages 2–4 in macaques, with longitudinal sampling) to characterize systematic developmental changes in amygdala–PFC functional connectivity patterns (diffuse-to-patchy/clustered, feedforward-to-feedback).'},
  {'id': 'PHEN_024',
   'title': 'Cortical Thinning and Synaptic Pruning',
   'justification': 'They report an age-related decline in the brainwide number of activated voxels from amygdala stimulation, explicitly interpreting this as mesoscale connectional pruning during early juvenile development.'},
  {'id': 'PHEN_006',
   'title': 'Hierarchical Information Flow (Effective Connectivity)',
   'justification': 'A core hypothesis concerns a developmental shift in directional influence between amygdala and PFC—from predominantly feedforward to predominantly feedback—i.e., changes in directed/hierarchical interactions rather than mere

In [456]:
cluster_info = []

for i, definition in enumerate(cluster_definitions):
    text = f"Cluster {i} - {definition['Title']}\nKeywords: {definition['Keywords']}\nDescription: {definition['Description']}"
    cluster_info.append(text)

cluster_text = '\n\n'.join(cluster_info)

required_fields = ["MetaClusters", "Assignments"]

chain_input = {'clusters': cluster_text}

meta_json2 = safe_dictionary_extraction(required_fields, chain_input, metacluster_chain, retries=retries, delay=delay)

In [457]:
meta_json

{'MetaClusters': [{'MetaTitle': 'Transdiagnostic Network Biomarkers in Psychiatric and Neurodevelopmental Disorders',
   'MetaDescription': 'This metacluster covers multimodal, network-based neuroimaging of major psychiatric and neurodevelopmental conditions, emphasizing large-scale connectivity, gradients, and structure–function coupling as biomarkers. Work here seeks to subtype individuals, model heterogeneity and trajectories, and predict symptoms and treatment response using normative and machine-learning frameworks across depression, psychosis, ADHD, autism, addictions, and early-environmental risk.',
   'InclusionCriteria': ['Primary focus on psychiatric or neurodevelopmental disorders',
    'Large-scale brain networks and connectivity as core explanatory constructs',
    'Multimodal imaging combined with advanced statistical or machine-learning models',
    'Emphasis on biomarkers, subtyping, normative modeling, or outcome prediction'],
   'MemberClusters': ['Cluster 0 - Network

In [458]:
meta_json2 = safe_dictionary_extraction(required_fields, chain_input, metacluster_chain, retries=retries, delay=delay)

In [459]:
meta_json2 # this one is better!

{'MetaClusters': [{'MetaTitle': 'Transdiagnostic Network Biomarkers of Psychiatric and Neurodevelopmental Disorders',
   'MetaDescription': 'This metacluster covers multimodal, network-centric neuroimaging studies of common psychiatric and neurodevelopmental conditions, emphasizing individual differences, subtyping, and prediction of clinical outcomes. Work combines structural, functional, neurochemical, and peripheral biological measures to derive biomarkers of symptom dimensions, risk states, and treatment response across mood, psychosis, anxiety, addiction, ADHD, and autism.',
   'InclusionCriteria': ['Primary focus on psychiatric or neurodevelopmental disorders',
    'Emphasis on large-scale brain networks and connectivity',
    'Multimodal biomarkers and normative/individualized modeling',
    'Links to symptoms, cognition, risk, and treatment outcomes'],
   'MemberClusters': ['Cluster 0 - Network- and Inflammation-Based Biomarkers of Depression and Mood Dysregulation from Multimo

In [536]:
import json 

met_cluster_file = os.path.join(BASEPATH, "metacluster_file.json")

with open(met_cluster_file, "w") as f:
    json.dump(meta_json2, f)

    

In [398]:
cluster = 0

cluster_posters = posters_2025[posters_2025['OHBM Cluster']==cluster].copy()
abstracts = cluster_posters['Abstract'].tolist()

abstracts_text = '\n'.join(abstracts)

chain_input = {'cluster_json': cluster_definitions[0],'abstracts': abstracts_text}

feedback = evaluation_chain.invoke(chain_input)

In [402]:
print(feedback.content)

The cluster description you were given captures many broad themes correctly (neurodegeneration, multimodal imaging, networks, machine learning, glymphatic/ALPS, tau vs amyloid, brain‑age, etc.), but it is too narrow and somewhat misleading for this particular set of abstracts. Below is targeted feedback on each requested element and how it could be improved.

---

### 1. Keywords

**Issue:**  
The keyword set is only partially aligned with this cluster. It emphasizes:

- “Alzheimer’s disease and related dementias; Parkinson’s disease and Lewy body disorders”
- “Multimodal neuroimaging biomarkers (structural, diffusion, PET, fMRI, EEG/MEG)”
- “Brain connectivity, networks, and gradients”
- “Machine learning, normative and progression modeling for early detection and prognosis”

These are indeed present, but they don’t reflect the full breadth or the *balance* of methods and diseases in this cluster.

**What’s missing / underweighted:**

- Other diseases:  
  - Frontotemporal dementia (b

In [503]:
""" 
"fit": "good|borderline|out of scope",
  "pipeline stage (primary)": "StandardsAndDataModels|ComputeAndReproducibility|AcquisitionAndProtocolDesign|ReconstructionAndLowLevelCorrection|PreprocessingAndFeatureExtraction|ModelingAndInference|PhysiologyAwareMeasurement",
  "pipeline stage (secondary)": "StandardsAndDataModels|ComputeAndReproducibility|AcquisitionAndProtocolDesign|ReconstructionAndLowLevelCorrection|PreprocessingAndFeatureExtraction|ModelingAndInference|PhysiologyAwareMeasurement|null",
  "contribution type (primary)": "ToolOrPipeline|BenchmarkOrComparison|MethodOrAlgorithm|ResourceOrDataset|TheoryOrBiophysicalModel|ValidationOrMetrology|ApplicationOrCaseStudy",
  "contribution type (secondary)": "ToolOrPipeline|BenchmarkOrComparison|MethodOrAlgorithm|ResourceOrDataset|TheoryOrBiophysicalModel|ValidationOrMetrology|ApplicationOrCaseStudy|null",
  "confidence": 0.00,
  "rationale": 
"""

gemma_llm = OllamaLLM(model="gemma2:9b")
tagging_chain = TAGGING_PROMPT | gemma_llm | SimpleJsonOutputParser()

required_fields = ['fit', 'pipeline_stage','contribution_type', 'confidence', 'rationale']

In [523]:
clusters = [2, 3, 10, 16]

abstract_tags = []
finished = set()   # use a set for O(1) lookup

for cluster_id in clusters:
    cluster_df = posters_2025[posters_2025['OHBM Cluster'] == cluster_id]

    for _, row in cluster_df.iterrows():
        abstract = row['Abstract']
        abstract_id = row['ID']

        if abstract_id in finished:
            continue

        chain_input = {'abstract': abstract}
        abstract_tag = safe_dictionary_extraction(
            required_fields,
            chain_input,
            tagging_chain,
            retries,
            delay
        )

        abstract_tags.append(abstract_tag)
        finished.add(abstract_id)


In [527]:

abstract_ids = []
for cluster_id in clusters:
    cluster_df = posters_2025[posters_2025['OHBM Cluster'] == cluster_id]

    for _, row in cluster_df.iterrows():
        abstract_id = row['ID']

        abstract_ids.append(abstract_id)

In [530]:
methods_tags_df = pd.DataFrame(abstract_tags)

methods_tags_df['ID'] = abstract_ids

In [534]:
methods_tags_file = os.path.join(BASEPATH, '2025_method_tagging.csv')

methods_tags_df.to_csv(met_cluster_file)

In [553]:
"""  
{
  "stance": "intrinsic_mechanism|epiphenomenon_artifact|mixed_or_conditional|no_position|not_relevant",
  "strength": 0,
  "evidence": ["short paraphrased phrase 1", "short paraphrased phrase 2"],
  "rationale": "1-2 sentences explaining the choice, grounded in the abstract"

"stance": "pro_criticality|anti_criticality|conditional_or_mixed|methods_only|no_position",
  "claim_type": ["operating_point","..."],
  "strength": 0,
  "evidence": ["short paraphrased cue 1","short paraphrased cue 2"],
  "rationale": "1–2 sentences grounded in the abstract"

"""


debate_chain = DEBATE_PROMPT | gemma_llm | SimpleJsonOutputParser()

required_fields = ['stance', 'claim_type', 'strength', 'evidence', 'rationale']

In [570]:
cluster_id = 1
cluster_df = posters_2025[posters_2025['OHBM Cluster'] == cluster_id]

debate_tags = []

finished = set()

for _, row in cluster_df.iterrows():
    abstract = row['Abstract']
    abstract_id = row['ID']

    if abstract_id in finished:
        continue

    chain_input = {'abstract': abstract}
    abstract_tag = safe_dictionary_extraction(
            required_fields,
            chain_input,
            debate_chain,
            retries,
            delay
        )
    print(abstract_tag)




{'stance': 'no_position', 'claim_type': [], 'strength': 0, 'evidence': [], 'rationale': ''}
{'stance': 'no_position', 'claim_type': [], 'strength': 0, 'evidence': [], 'rationale': ''}
{'stance': 'no_position', 'claim_type': [], 'strength': 0, 'evidence': [], 'rationale': 'The abstract focuses on the impact of subcortical infarction on functional connectivity and cortical hierarchy organization, without mentioning criticality or related concepts.'}
{'stance': 'no_position', 'claim_type': [], 'strength': 0, 'evidence': [], 'rationale': ''}
{'stance': 'no_position', 'claim_type': [], 'strength': 0, 'evidence': [], 'rationale': "The abstract focuses on identifying shared brain networks across cognitive domains (memory, semantics, mentalizing) using resting-state functional connectivity. It doesn't discuss criticality or its potential role."}
{'stance': 'no_position', 'claim_type': [], 'strength': 0, 'evidence': [], 'rationale': ''}
{'stance': 'no_position', 'claim_type': [], 'strength': 0,

In [568]:
print(abstract)

Neurological conditions such as stroke and glioma are major causes of disability worldwide, accounting for 
millions of deaths per year and long-lasting cognitive impairments1. Recent studies show that brain network disruption can 
predict survival rates in glioma and model cognitive impairment in stroke2,3. However, a similar connectivity framework to 
model cognitive functioning after surgery is currently lacking. Here, we introduce a novel method that integrates direct 
electrical brain stimulation (DES) with MRI-driven functional network mapping to recover the white matter substrates causally 
implicated in language production.
Methods: Recent evidence suggests that spontaneous hemodynamic oscillations in the white matter map into distributed 
functional brain networks with distinct neurophysiological underpinnings4-6. In line with these observations, we used white 
matter DES points causing transient speech arrest, semantic or phonological aphasia (N=297 patients, 486 stimulation

In [41]:
poster_df = pd.read_csv(os.path.join(BASEPATH, "2025/ohbm_posters_2025.csv"))

In [44]:
print(poster_df[poster_df['ID']==811538240801]['Abstract'].values[0])

Replication is a cornerstone of scientific research, but achieving robust results that can be replicated remains 
a challenge in neuroimaging. The problem stems from the high degree of flexibility in data processing and analysis choices, 
known as the “researcher’s degrees of freedom”, which can affect scientific results. Multiverse analysis addresses this 
problem by systematically exploring the impact of alternative analytical choices on the results of hypothesis tests. By running 
analyses through a range of defensible pipeline options, multiverse analysis tests robustness, a core aspect of replicability. 
This study examines the robustness of findings on how prematurity affects neonatal brain connectivity using multiverse 
analysis. The focus is on the audio-visual integration (AVI) brain network, a key system for perceptual and ultimately socio-
cognitive development. Using fMRI data from the Developing Human Connectome Project (dHCP), we tested the robustness 
of the association 